# Hidden Pairs — воспроизведение выбранного SVM

Командное решение Chess Monsters: Ермолаев, Камалов, Байрамов.

Это короткий воспроизводимый вариант выбранной конфигурации из `svm_final-ru.ipynb`, без повторного поиска гиперпараметров. Используются все 512 исходных признаков, без масштабирования и PCA. Обучение и калибровка выполняются только на train; validation используется для оценки. Полученный CSV — результат нового запуска, а не архивный файл отправки на Kaggle.

In [1]:
from pathlib import Path
import time
import platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

SEED = 20260916
DATA_DIR = Path('.')
OUTPUT_DIR = DATA_DIR / 'runs' / ('final-' + time.strftime('%Y%m%d-%H%M%S'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print('Python:', platform.python_version(), 'scikit-learn:', sklearn.__version__)

Python: 3.14.3 scikit-learn: 1.8.0


In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
validation = pd.read_csv(DATA_DIR / 'validation.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
features = [c for c in train.columns if c not in {'row_id', 'target'}]
assert len(features) == 512
assert features == [c for c in validation.columns if c not in {'row_id', 'target'}]
assert features == [c for c in test.columns if c != 'row_id']
assert (len(train), len(validation), len(test)) == (2520, 560, 1116)
for frame in (train, validation, test):
    assert np.isfinite(frame[features].to_numpy()).all()
for frame in (train, validation):
    assert set(frame.target.unique()) == {0, 1}
assert list(sample.columns) == ['row_id', 'target']
assert sample.row_id.equals(test.row_id)
assert sample.row_id.is_unique

## Обучение и калибровка

RBF-SVM: `C=2.5`, `gamma=0.007`. Sigmoid-калибровка преобразует оценки SVM в вероятности. При `ensemble=False` калибратор обучается на out-of-fold оценках train, а итоговый SVM — на всём train. Метки validation в обучение не передаются.

In [3]:
model = CalibratedClassifierCV(
    estimator=SVC(kernel='rbf', C=2.5, gamma=0.007,
                  cache_size=512, random_state=SEED),
    method='sigmoid',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    ensemble=False,
    n_jobs=1,
)
started = time.perf_counter()
model.fit(train[features], train.target)
validation_probability = model.predict_proba(validation[features])[:, 1]
validation_auc = roc_auc_score(validation.target, validation_probability)
elapsed = time.perf_counter() - started
print(f'Validation ROC-AUC: {validation_auc:.6f}')
print(f'Fit, calibration and validation: {elapsed:.2f} s')
assert np.isfinite(validation_auc)
assert np.isclose(validation_auc, 0.947921, atol=1e-6)

Validation ROC-AUC: 0.947921
Fit, calibration and validation: 15.17 s


In [4]:
submission = sample.copy()
submission['target'] = model.predict_proba(test[features])[:, 1]
assert submission.shape == (1116, 2)
assert submission.row_id.equals(test.row_id)
assert submission.row_id.is_unique
assert np.isfinite(submission.target).all()
assert submission.target.between(0, 1).all()
submission.to_csv(OUTPUT_DIR / 'submission_svm.csv', index=False)
print('Saved:', OUTPUT_DIR / 'submission_svm.csv')
submission.head()

Saved: runs\final-20261001-211451\submission_svm.csv


,row_id,target
0,row_46ad1afad8562239f79c,0.965567
1,row_b42ade9cd57b8d919acb,0.921229
2,row_c951cd5e197b6b4e8b83,0.266900
3,row_a54b3423760d599416ce,0.063705
4,row_a87c87cc47690e2a303f,0.994720
